# Phase 36 — T5 Text-to-Text Classification

Χρήση του T5 (Text-to-Text Transfer Transformer) για Food Hazard Detection.

**Διαφορά από BERT:**
- BERT: encoder-only → classification head → label
- T5: encoder-decoder → generates text → label as text

**Φορμάτ input/output:**
```
Input:  'classify hazard: <title> <text>'
Output: 'biological'

Input:  'classify product: <title> <text>'
Output: 'meat, egg and dairy products'
```

**Μοντέλο:** t5-small (60M params)

**Σκοπός:** Σύγκριση encoder-only (BERT) vs encoder-decoder (T5)

In [ ]:
import torch
import numpy as np
import pandas as pd
import copy
from torch.utils.data import Dataset, DataLoader
from transformers import T5ForConditionalGeneration, T5Tokenizer, get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.metrics import f1_score
import warnings
warnings.filterwarnings('ignore')

import random
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
train = pd.read_csv('/train.csv')
valid = pd.read_csv('/valid.csv')
test  = pd.read_csv('/test.csv')

def make_text(df):
    return (df['title'].fillna('') + ' ' + df['text'].fillna('').str[:550]).tolist()

texts_train = make_text(train)
texts_valid = make_text(valid)
texts_test  = make_text(test)

# Labels ως strings (T5 παράγει text)
y_train_hazard  = train['hazard-category'].tolist()
y_train_product = train['product-category'].tolist()
y_valid_hazard  = valid['hazard-category'].tolist()
y_valid_product = valid['product-category'].tolist()

all_hazard_labels  = sorted(train['hazard-category'].unique())
all_product_labels = sorted(pd.concat([train['product-category'], valid['product-category']]).unique())

print(f'Train: {len(train)} | Valid: {len(valid)} | Test: {len(test)}')
print(f'Hazard labels ({len(all_hazard_labels)}): {all_hazard_labels[:3]}...')
print(f'Product labels ({len(all_product_labels)}): {all_product_labels[:3]}...')

In [ ]:
MODEL_NAME = 't5-small'
MAX_INPUT_LEN  = 128   # Input tokens
MAX_TARGET_LEN = 16    # Output tokens (label είναι μικρό)
BATCH_SIZE     = 16
LR             = 3e-4  # T5 χρειάζεται μεγαλύτερο LR από BERT
N_EPOCHS       = 10
WARMUP_RATIO   = 0.1

print(f'Φόρτωση T5 tokenizer: {MODEL_NAME}')
tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME)

In [ ]:
class T5Dataset(Dataset):
    """
    Dataset για T5 text-to-text classification.
    Input:  'classify hazard: <text>'
    Target: '<label>'
    """
    def __init__(self, texts, labels, tokenizer, task_prefix,
                 max_input_len=128, max_target_len=16):
        self.texts          = texts
        self.labels         = labels
        self.tokenizer      = tokenizer
        self.task_prefix    = task_prefix
        self.max_input_len  = max_input_len
        self.max_target_len = max_target_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        # Input: task prefix + text
        input_text = f'{self.task_prefix}: {self.texts[idx]}'
        input_enc  = self.tokenizer(
            input_text,
            max_length=self.max_input_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        # Target: label as text
        target_enc = self.tokenizer(
            str(self.labels[idx]),
            max_length=self.max_target_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        # T5 χρειάζεται -100 για padding στο target
        labels = target_enc['input_ids'].squeeze()
        labels[labels == self.tokenizer.pad_token_id] = -100

        return {
            'input_ids':      input_enc['input_ids'].squeeze(),
            'attention_mask': input_enc['attention_mask'].squeeze(),
            'labels':         labels
        }


print('T5Dataset defined')
print(f'\nΠαράδειγμα input: "classify hazard: {texts_train[0][:60]}..."')
print(f'Παράδειγμα target: "{y_train_hazard[0]}"')

In [ ]:
def train_t5(texts_train, labels_train, texts_valid, labels_valid,
             task_prefix, all_labels, task_name):
    print(f'\n=== ΕΚΠΑΙΔΕΥΣΗ T5 — {task_name} ===')

    train_loader = DataLoader(
        T5Dataset(texts_train, labels_train, tokenizer, task_prefix,
                  MAX_INPUT_LEN, MAX_TARGET_LEN),
        batch_size=BATCH_SIZE, shuffle=True
    )
    valid_loader = DataLoader(
        T5Dataset(texts_valid, labels_valid, tokenizer, task_prefix,
                  MAX_INPUT_LEN, MAX_TARGET_LEN),
        batch_size=BATCH_SIZE, shuffle=False
    )

    model     = T5ForConditionalGeneration.from_pretrained(MODEL_NAME).to(device)
    optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    total_steps  = len(train_loader) * N_EPOCHS
    warmup_steps = int(total_steps * WARMUP_RATIO)
    scheduler = get_linear_schedule_with_warmup(
        optimizer, warmup_steps, total_steps
    )
    print(f'Total steps: {total_steps} | Warmup: {warmup_steps}\n')

    best_f1    = 0
    best_state = None
    patience   = 3
    patience_cnt = 0

    for epoch in range(N_EPOCHS):
        # Training
        model.train()
        total_loss = 0
        for batch in train_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            optimizer.zero_grad()
            loss = model(**batch).loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            total_loss += loss.item()

        # Validation — generate predictions
        model.eval()
        preds = []
        with torch.no_grad():
            for batch in valid_loader:
                input_ids      = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                # Generate output tokens
                outputs = model.generate(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    max_new_tokens=MAX_TARGET_LEN,
                    num_beams=1  # Greedy decoding για ταχύτητα
                )
                # Decode tokens → text
                decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
                # Snap to nearest valid label
                for pred in decoded:
                    pred = pred.strip().lower()
                    if pred in all_labels:
                        preds.append(pred)
                    else:
                        # Βρίσκουμε το πιο κοντινό label
                        best_match = min(all_labels, key=lambda l: abs(len(l) - len(pred)))
                        preds.append(best_match)

        f1 = f1_score(labels_valid, preds, average='macro', zero_division=0)
        print(f'Epoch {epoch+1}/{N_EPOCHS} | Loss: {total_loss/len(train_loader):.4f} | F1: {f1:.4f}')

        if f1 > best_f1:
            best_f1      = f1
            best_state   = copy.deepcopy(model.state_dict())
            patience_cnt = 0
        else:
            patience_cnt += 1
            print(f' (patience {patience_cnt}/{patience})')
            if patience_cnt >= patience:
                print('Early stopping')
                break

    model.load_state_dict(best_state)
    print(f'\nΚαλύτερο F1: {best_f1:.4f}')
    return model


def get_t5_predictions(model, texts, task_prefix, all_labels):
    model.eval()
    dummy_labels = [''] * len(texts)
    loader = DataLoader(
        T5Dataset(texts, dummy_labels, tokenizer, task_prefix,
                  MAX_INPUT_LEN, MAX_TARGET_LEN),
        batch_size=BATCH_SIZE, shuffle=False
    )
    preds = []
    with torch.no_grad():
        for batch in loader:
            outputs = model.generate(
                input_ids=batch['input_ids'].to(device),
                attention_mask=batch['attention_mask'].to(device),
                max_new_tokens=MAX_TARGET_LEN,
                num_beams=1
            )
            decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
            for pred in decoded:
                pred = pred.strip().lower()
                if pred in all_labels:
                    preds.append(pred)
                else:
                    best_match = min(all_labels, key=lambda l: abs(len(l) - len(pred)))
                    preds.append(best_match)
    return preds

## Εκπαίδευση T5 — Hazard

In [ ]:
t5_hazard = train_t5(
    texts_train, y_train_hazard,
    texts_valid, y_valid_hazard,
    task_prefix='classify hazard',
    all_labels=all_hazard_labels,
    task_name='HAZARD'
)

## Εκπαίδευση T5 — Product

In [ ]:
t5_product = train_t5(
    texts_train, y_train_product,
    texts_valid, y_valid_product,
    task_prefix='classify product',
    all_labels=all_product_labels,
    task_name='PRODUCT'
)

## Αξιολόγηση & Submission

In [ ]:
def official_st1_score(y_true_h, y_pred_h, y_true_p, y_pred_p, verbose=True):
    f1_h = f1_score(y_true_h, y_pred_h, average='macro', zero_division=0)
    mask = (np.array(y_true_h) == np.array(y_pred_h))
    f1_p = f1_score(
        np.array(y_true_p)[mask], np.array(y_pred_p)[mask],
        average='macro', zero_division=0
    ) if mask.sum() > 0 else 0.0
    score = (f1_h + f1_p) / 2
    if verbose:
        print(f'  macro-F1 Hazard:                    {f1_h:.4f}')
        print(f'  Σωστά hazard:                       {mask.sum()}/{len(mask)} ({100*mask.mean():.1f}%)')
        print(f'  macro-F1 Product (given correct h): {f1_p:.4f}')
        print(f'  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
        print(f'  OFFICIAL ST1 SCORE:                 {score:.4f}')
    return score

# Validation
pred_hazard_valid  = get_t5_predictions(t5_hazard,  texts_valid, 'classify hazard',  all_hazard_labels)
pred_product_valid = get_t5_predictions(t5_product, texts_valid, 'classify product', all_product_labels)

print('=== ΑΞΙΟΛΟΓΗΣΗ T5-small (validation) ===')
score_t5 = official_st1_score(
    y_valid_hazard, pred_hazard_valid,
    y_valid_product, pred_product_valid
)

print('\n=== ΣΥΓΚΡΙΣΗ ARCHITECTURES ===')
print(f'TextCNN:       0.6280  (Kaggle)')
print(f'BiLSTM:        0.58579 (Kaggle)')
print(f'T5-small:      {score_t5:.4f}  (validation)')
print(f'DistilBERT:    0.7606  (Kaggle)')
print(f'BERT-base:     0.8040  (Kaggle)')
print(f'Best ensemble: 0.8188  (Kaggle)')

In [ ]:
# Test predictions
print('Παραγωγή test predictions')
test_hazard  = get_t5_predictions(t5_hazard,  texts_test, 'classify hazard',  all_hazard_labels)
test_product = get_t5_predictions(t5_product, texts_test, 'classify product', all_product_labels)

pd.DataFrame({
    'id': test['id'],
    'hazard-category': test_hazard,
    'product-category': test_product
}).to_csv('submission_t5.csv', index=False)

print('Αποθηκεύτηκε: submission_t5.csv')
print(f'Validation score: {score_t5:.4f}')